In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CONTROL_TABLE = (
    "target_catalog.control.ingestion_control"
)

# Select only active tables configured for FULL load
control_rows = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("active") == True)
        & (F.upper(F.col("load_type")) == "FULL")
    )
    .collect()
)

print(f"Full-load tables to process: {len(control_rows)}")

control_delta = DeltaTable.forName(
    spark, CONTROL_TABLE
)

for config in control_rows:

    table_id = config["table_id"]
    source_table = config["source_table"]
    target_table = config["target_table"]
    primary_key = config["primary_key"]
    watermark_column = config["watermark_column"]

    print(f"\nStarting FULL LOAD: {table_id}")

    try:
        # 1. Read the complete source table
        source_df = spark.table(source_table)

        # 2. Remove the Auto Loader operational column
        if "_rescued_data" in source_df.columns:
            source_df = source_df.drop("_rescued_data")

        # 3. Deduplicate by primary key, keeping latest version
        window_spec = (
            Window
            .partitionBy(primary_key)
            .orderBy(
                F.col(watermark_column).desc_nulls_last()
            )
        )

        full_df = (
            source_df
            .withColumn(
                "_rn",
                F.row_number().over(window_spec)
            )
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

        # 4. Validate that the source has records
        row_count = full_df.count()

        if row_count == 0:
            raise ValueError(
                f"Full load stopped for {table_id}: "
                "source is empty. Target was not replaced."
            )

        # 5. Validate the primary key
        null_key_count = (
            full_df
            .filter(F.col(primary_key).isNull())
            .count()
        )

        if null_key_count > 0:
            raise ValueError(
                f"{table_id} has {null_key_count} null primary keys"
            )

        # 6. Replace the complete target snapshot
        (
            full_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        # 7. Get the latest source watermark
        max_watermark = (
            full_df
            .agg(
                F.max(watermark_column).alias("max_value")
            )
            .first()["max_value"]
        )

        # 8. Update the control table
        control_delta.update(
            condition=F.col("table_id") == table_id,
            set={
                "last_processed_value":
                    F.lit(max_watermark).cast("timestamp"),
                "last_run_at": F.current_timestamp(),
                "rows_processed":
                    F.lit(row_count).cast("bigint"),
                "status": F.lit("SUCCESS"),
                "error_message": F.lit(None).cast("string")
            }
        )

        print(
            f"{table_id}: FULL LOAD SUCCESS; "
            f"{row_count} rows written to {target_table}"
        )

    except Exception as e:

        error_text = str(e)[:4000]

        control_delta.update(
            condition=F.col("table_id") == table_id,
            set={
                "last_run_at": F.current_timestamp(),
                "rows_processed": F.lit(0).cast("bigint"),
                "status": F.lit("FAILED"),
                "error_message": F.lit(error_text)
            }
        )

        print(f"{table_id}: FULL LOAD FAILED - {error_text}")
        raise

print("\nFull-load processing completed.")